# EE 446 Homework 1 Programming Notebook

Use the **tinyml-arduino** Python environment that you set up for this class. In JupyterLab, select the kernel named **Python (tinyml-arduino)** before running this notebook.

Do not install or uninstall TensorFlow packages inside this notebook. The class environment already contains the required packages for this assignment, including TensorFlow, TensorFlow Model Optimization Toolkit, scikit-learn, NumPy, pandas, and JupyterLab.

This notebook contains the programming questions marked **[Pro]**. Complete each section by replacing the placeholder comments with your own code. Print the requested outputs so that your work can be graded directly from the notebook.


In [1]:
import sys
print(sys.executable)

/Users/kourosh/ai/projects/tinyml-arduino/bin/python


In [2]:
import sys
!{sys.executable} -m pip install "tensorflow-model-optimization==0.8.0"

In [3]:
import sys
!{sys.executable} -m pip install "keras==2.14.0"

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, r2_score

import tensorflow as tf
import tensorflow_model_optimization as tfmot

Sequential = tf.keras.Sequential
Dense = tf.keras.layers.Dense
LSTM = tf.keras.layers.LSTM
to_categorical = tf.keras.utils.to_categorical

print("TensorFlow version:", tf.__version__)
print("TF-MOT version:", tfmot.__version__)

TensorFlow version: 2.14.1
TF-MOT version: 0.8.0



---

# Problem 1: DNN and Wine Classification (80 points)

This problem uses the Wine dataset available through scikit-learn. The dataset is loaded locally from the installed package, so no external data file is required.


In [6]:
# Load the Wine dataset from scikit-learn.
# This avoids requiring an external wine.data file.

wine = load_wine(as_frame=True)

feature_names = list(wine.feature_names)
df = wine.frame.copy()
df["Class"] = wine.target

# Reorder the columns so that the class label appears first.
df = df[["Class"] + feature_names]

# Number of classes
num_classes = df["Class"].nunique()
print("Number of classes:", num_classes)

# Number of features, excluding the class label
num_features = df.shape[1] - 1
print("Number of features:", num_features)

# Basic feature statistics
feature_stats = df.drop(columns=["Class"]).describe().T[["min", "max", "mean", "std"]]
print("\nFeature statistics:\n", feature_stats)

# Class distribution
class_counts = df["Class"].value_counts().sort_index()
print("\nClass distribution:\n", class_counts)


Number of classes: 3
Number of features: 13

Feature statistics:
                                  min      max        mean         std
alcohol                        11.03    14.83   13.000618    0.811827
malic_acid                      0.74     5.80    2.336348    1.117146
ash                             1.36     3.23    2.366517    0.274344
alcalinity_of_ash              10.60    30.00   19.494944    3.339564
magnesium                      70.00   162.00   99.741573   14.282484
total_phenols                   0.98     3.88    2.295112    0.625851
flavanoids                      0.34     5.08    2.029270    0.998859
nonflavanoid_phenols            0.13     0.66    0.361854    0.124453
proanthocyanins                 0.41     3.58    1.590899    0.572359
color_intensity                 1.28    13.00    5.058090    2.318286
hue                             0.48     1.71    0.957449    0.228572
od280/od315_of_diluted_wines    1.27     4.00    2.611685    0.709990
proline                 

## Problem 1 - Part (a)
### Base Model Training and Evaluation


In [7]:
# Step 1: Separate the feature matrix and class labels.
# - Assign the feature columns to variable X.
# - Assign the class labels to variable y.
# - The labels in this scikit-learn dataset are already zero-based: 0, 1, and 2.

# <-- Enter your code here <--#

X = df.drop(columns=["Class"]).values
y = df["Class"].values

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Unique labels:", np.unique(y))

X shape: (178, 13)
y shape: (178,)
Unique labels: [0 1 2]


In [8]:
# Step 2: Perform a train-test split (70% train, 30% test) using random_state=42

# <-- Enter your code here <--#
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (124, 13)
X_test shape: (54, 13)


In [9]:
# Step 3: Use StandardScaler to normalize the features
# - Fit on X_train and transform both X_train and X_test

# <-- Enter your code here <--#
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Train mean (first 3 features):", X_train_scaled.mean(axis=0)[:3])
print("Train std  (first 3 features):", X_train_scaled.std(axis=0)[:3])

Train mean (first 3 features): [3.08982230e-15 3.23218155e-16 2.34579381e-15]
Train std  (first 3 features): [1. 1. 1.]


In [10]:
# Step 4: Use one-hot encoding for y_train and y_test.
# - Use tf.keras.utils.to_categorical.
# - Use num_classes=num_classes to make the output shape explicit.

# <-- Enter your code here <--#
y_train_cat = to_categorical(y_train, num_classes=num_classes)
y_test_cat = to_categorical(y_test, num_classes=num_classes)

print("y_train_cat shape:", y_train_cat.shape)
print("y_test_cat shape:", y_test_cat.shape)

y_train_cat shape: (124, 3)
y_test_cat shape: (54, 3)


In [12]:
# Step 5: Define a Sequential model with the following architecture:
# - Dense(64, activation='relu')
# - Dense(32, activation='relu')
# - Dense(num_classes, activation='softmax')
# Make sure the first Dense layer receives the correct input shape.

# <-- Enter your code here <--#
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(32, activation='relu'),
    Dense(num_classes, activation='softmax')
])
model.summary()

Model: "sequential_1"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_3 (Dense)             (None, 64)                896       
                                                                 
 dense_4 (Dense)             (None, 32)                2080      
                                                                 
 dense_5 (Dense)             (None, 3)                 99        
                                                                 
Total params: 3075 (12.01 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [13]:
# Step 6: Compile using Adam optimizer, categorical_crossentropy loss, and accuracy metric
# - Train for 20 epochs with batch_size=8 and validation_split=0.2

# <-- Enter your code here <--#
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train_scaled, y_train_cat,
    epochs=20,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/20
13/13 [==============================] - 0s 6ms/step - loss: 0.8735 - accuracy: 0.6465 - val_loss: 0.7722 - val_accuracy: 0.7600
Epoch 2/20
13/13 [==============================] - 0s 1ms/step - loss: 0.5859 - accuracy: 0.8485 - val_loss: 0.5256 - val_accuracy: 0.9200
Epoch 3/20
13/13 [==============================] - 0s 1ms/step - loss: 0.4112 - accuracy: 0.9293 - val_loss: 0.3792 - val_accuracy: 0.9200
Epoch 4/20
13/13 [==============================] - 0s 1ms/step - loss: 0.2924 - accuracy: 0.9798 - val_loss: 0.2721 - val_accuracy: 0.9600
Epoch 5/20
13/13 [==============================] - 0s 1ms/step - loss: 0.2127 - accuracy: 0.9798 - val_loss: 0.2032 - val_accuracy: 0.9600
Epoch 6/20
13/13 [==============================] - 0s 1ms/step - loss: 0.1569 - accuracy: 0.9899 - val_loss: 0.1625 - val_accuracy: 0.9600
Epoch 7/20
13/13 [==============================] - 0s 1ms/step - loss: 0.1179 - accuracy: 0.9899 - val_loss: 0.1356 - val_accuracy: 0.9600
Epoch 8/20
13/13 [==

In [14]:
# Step 7: Evaluate the model on test data and print:
# - Accuracy
# - Classification report
# - Confusion matrix

# <-- Enter your code here <--#
train_loss, train_acc = model.evaluate(X_train_scaled, y_train_cat, verbose=0)
test_loss, test_acc = model.evaluate(X_test_scaled, y_test_cat, verbose=0)

print(f"Training accuracy: {train_acc:.4f}")
print(f"Test accuracy    : {test_acc:.4f}")

y_pred = np.argmax(model.predict(X_test_scaled, verbose=0), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

print("\nClassification report:\n", classification_report(y_true, y_pred))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))

Training accuracy: 0.9919
Test accuracy    : 0.9815

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


In [15]:
# Step 8: Convert the trained model to TFLite format and save it as "model_base.tflite"
# - Print the file size in kilobytes

# <-- Enter your code here <--#
import os

def file_size_kb(filename):
    """Return the size of a file in kilobytes."""
    return os.path.getsize(filename) / 1024


converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_base = converter.convert()

with open("model_base.tflite", "wb") as f:
    f.write(tflite_base)

print(f"Baseline float32 TFLite model size: {file_size_kb('model_base.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpg3immkoh/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpg3immkoh/assets


Baseline float32 TFLite model size: 14.10 KB


2026-07-24 18:25:45.918606: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:25:45.918624: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:25:45.919021: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpg3immkoh
2026-07-24 18:25:45.919430: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:25:45.919435: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpg3immkoh
2026-07-24 18:25:45.921109: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:382] MLIR V1 optimization pass is not enabled
2026-07-24 18:25:45.921486: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-24 18:25:45.941519: I tensorflow/cc/saved_model/loader.

## Problem 1 - Part (b)

### Quantization (int8, float16, dynamic range)


In [16]:
def representative_data_gen(X_reference, num_samples=100):
    """Create a representative dataset generator for full integer quantization."""
    max_samples = min(num_samples, len(X_reference))
    for i in range(max_samples):
        yield [X_reference[i:i + 1].astype(np.float32)]


def quantize_and_evaluate(model, X_test, y_test_cat, quant_type, filename):
    """Convert a Keras model to TFLite, evaluate it, and report model size.

    Parameters
    ----------
    model : tf.keras.Model
        Trained Keras model.
    X_test : np.ndarray
        Test features after the same preprocessing used for training.
    y_test_cat : np.ndarray
        One-hot encoded test labels.
    quant_type : str
        One of: 'int8', 'float16', or 'dynamic'.
    filename : str
        Output TFLite filename.
    """

    # Create the TFLite converter from the trained Keras model.
    converter = tf.lite.TFLiteConverter.from_keras_model(model)

    # Step 1: Apply quantization settings.
    if quant_type == 'int8':
        # (a) Enable default optimizations.
        # (b) Provide representative_data_gen(X_train_scaled).
        # (c) Set supported_ops to TFLITE_BUILTINS_INT8.
        # (d) Set inference_input_type and inference_output_type to tf.int8.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
        converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
        converter.inference_input_type = tf.int8
        converter.inference_output_type = tf.int8

    elif quant_type == 'float16':
        # (a) Enable default optimizations.
        # (b) Set supported_types to [tf.float16].

        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        converter.target_spec.supported_types = [tf.float16]

    elif quant_type == 'dynamic':
        # (a) Enable default optimizations.
        converter.optimizations = [tf.lite.Optimize.DEFAULT]

    else:
        raise ValueError("quant_type must be one of: 'int8', 'float16', or 'dynamic'.")

    # Step 2: Convert the model and save it to the provided filename.

    tflite_model = converter.convert()
    with open(filename, "wb") as f:
        f.write(tflite_model)

    # Step 3: Run TFLite inference.
    # Complete the following:
    # - Use tf.lite.Interpreter to load the TFLite model.
    # - Allocate tensors.
    # - Get input and output tensor details.
    # - If the input is quantized, quantize each test sample using scale and zero point.
    # - If the output is quantized, dequantize the prediction using scale and zero point.
    # - Collect predictions into y_pred using np.argmax.
    # - Compare with y_true = np.argmax(y_test_cat, axis=1).

    interpreter = tf.lite.Interpreter(model_path=filename)
    interpreter.allocate_tensors()

    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]

    y_pred = []
    for i in range(len(X_test)):
        sample = X_test[i:i + 1].astype(np.float32)

        # If the input is quantized, quantize the sample using scale and zero point.
        if input_details["dtype"] == np.int8:
            input_scale, input_zero_point = input_details["quantization"]
            sample = sample / input_scale + input_zero_point
            sample = np.round(sample).astype(np.int8)

        interpreter.set_tensor(input_details["index"], sample)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details["index"])[0]

        # If the output is quantized, dequantize the prediction using scale and zero point.
        if output_details["dtype"] == np.int8:
            output_scale, output_zero_point = output_details["quantization"]
            output = (output.astype(np.float32) - output_zero_point) * output_scale

        y_pred.append(np.argmax(output))

    y_pred = np.array(y_pred)
    y_true = np.argmax(y_test_cat, axis=1)

    # Step 4: Report results.
    print(f"\n{quant_type.upper()} TFLite model size: {file_size_kb(filename):.2f} KB")

    accuracy = np.mean(y_pred == y_true)
    print(f"{quant_type.upper()} accuracy: {accuracy:.4f}")
    print("\nClassification report:\n", classification_report(y_true, y_pred))
    print("Confusion matrix:\n", confusion_matrix(y_true, y_pred))


In [17]:
# Step 5: Use the function above to create and evaluate three quantized models:
# - 'int8' saved as 'model_int8.tflite'
# - 'float16' saved as 'model_float16.tflite'
# - 'dynamic' saved as 'model_dynamic.tflite'

quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'int8', 'model_int8.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'float16', 'model_float16.tflite')
quantize_and_evaluate(model, X_test_scaled, y_test_cat, 'dynamic', 'model_dynamic.tflite')


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmp9tf6c7es/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmp9tf6c7es/assets



INT8 TFLite model size: 5.76 KB
INT8 accuracy: 0.9815

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


/Users/kourosh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-07-24 18:32:02.453530: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:32:02.453540: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:32:02.453633: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmp9tf6c7es
2026-07-24 18:32:02.454095: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:32:02.454099: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmp9tf6c7es
2026-07-24 18:32:02.455317: I tensorflow/cc/saved_model/loader.cc:233] 

INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpzghxirzn/assets


...

2026-07-24 18:32:02.741035: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:32:02.741046: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:32:02.741138: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpzghxirzn
2026-07-24 18:32:02.741571: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:32:02.741575: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpzghxirzn
2026-07-24 18:32:02.742629: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-24 18:32:02.758402: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpzghxirzn
2026-07-

INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpef_8pvrh/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpef_8pvrh/assets



DYNAMIC TFLite model size: 8.20 KB
DYNAMIC accuracy: 0.9815

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      0.95      0.98        21
           2       0.94      1.00      0.97        15

    accuracy                           0.98        54
   macro avg       0.98      0.98      0.98        54
weighted avg       0.98      0.98      0.98        54

Confusion matrix:
 [[18  0  0]
 [ 0 20  1]
 [ 0  0 15]]


2026-07-24 18:32:02.982821: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:32:02.982835: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:32:02.982927: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpef_8pvrh
2026-07-24 18:32:02.983356: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:32:02.983360: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpef_8pvrh
2026-07-24 18:32:02.984489: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-24 18:32:03.001404: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpef_8pvrh
2026-07-

## Problem 1 - Part (c)

### Pruning

In [18]:
# Step 1: Define a pruning schedule using tfmot.sparsity.keras.PolynomialDecay
# HINT:
# - Use initial_sparsity = 0.5 and final_sparsity = 0.7
# - Set end_step to total training steps (approx. dataset_size / batch_size * epochs)

batch_size = 8
epochs_prune = 10
end_step = int(np.ceil(len(X_train_scaled) / batch_size)) * epochs_prune

pruning_schedule = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step
)

print("End step:", end_step)

End step: 160


In [19]:
# Step 2: Build a Sequential model with 3 pruned Dense layers:
# - Dense(64, relu)
# - Dense(32, relu)
# - Dense(3, softmax)
# Make sure each Dense layer is wrapped with prune_low_magnitude()

prune_low_magnitude = tfmot.sparsity.keras.prune_low_magnitude

pruned_model = Sequential([
    prune_low_magnitude(
        Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        Dense(32, activation='relu'),
        pruning_schedule=pruning_schedule
    ),
    prune_low_magnitude(
        Dense(3, activation='softmax'),
        pruning_schedule=pruning_schedule
    )
])

pruned_model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 prune_low_magnitude_dense_  (None, 64)                1730      
 6 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 32)                4130      
 7 (PruneLowMagnitude)                                           
                                                                 
 prune_low_magnitude_dense_  (None, 3)                 197       
 8 (PruneLowMagnitude)                                           
                                                                 
Total params: 6057 (23.67 KB)
Trainable params: 3075 (12.01 KB)
Non-trainable params: 2982 (11.66 KB)
_________________________________________________________________


In [20]:
# Step 3: Compile the model with categorical_crossentropy and accuracy
# - Train for 10 epochs with batch_size=8 and validation_split=0.2
# - Add tfmot.sparsity.keras.UpdatePruningStep() to the callbacks list

pruned_model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [tfmot.sparsity.keras.UpdatePruningStep()]

history_pruned = pruned_model.fit(
    X_train_scaled, y_train_cat,
    epochs=epochs_prune,
    batch_size=batch_size,
    validation_split=0.2,
    callbacks=callbacks,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 1s 5ms/step - loss: 0.9507 - accuracy: 0.5556 - val_loss: 0.7863 - val_accuracy: 0.8000
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 0.7127 - accuracy: 0.8586 - val_loss: 0.5748 - val_accuracy: 0.8800
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.5309 - accuracy: 0.9394 - val_loss: 0.4180 - val_accuracy: 0.9600
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.3890 - accuracy: 0.9697 - val_loss: 0.3035 - val_accuracy: 0.9600
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.2777 - accuracy: 0.9899 - val_loss: 0.2189 - val_accuracy: 1.0000
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.1955 - accuracy: 0.9899 - val_loss: 0.1637 - val_accuracy: 1.0000
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.1413 - accuracy: 0.9899 - val_loss: 0.1289 - val_accuracy: 1.0000
Epoch 8/10
13/13 [==

In [21]:
# Step 4: Remove pruning wrappers using tfmot.sparsity.keras.strip_pruning().
# Then convert the stripped model to TFLite and save it as "model_pruned.tflite".
# Print the final file size in KB.

# Important: converting the unstripped pruned model can keep extra pruning variables
# and make the saved model larger than expected.

stripped_model = tfmot.sparsity.keras.strip_pruning(pruned_model)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_model)
tflite_pruned = converter.convert()

with open("model_pruned.tflite", "wb") as f:
    f.write(tflite_pruned)

print(f"Pruned TFLite model size: {file_size_kb('model_pruned.tflite'):.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpx2jhcetk/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpx2jhcetk/assets


Pruned TFLite model size: 14.14 KB


2026-07-24 18:45:27.806282: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:45:27.806292: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:45:27.806371: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpx2jhcetk
2026-07-24 18:45:27.806642: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:45:27.806645: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpx2jhcetk
2026-07-24 18:45:27.807237: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-24 18:45:27.814258: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpx2jhcetk
2026-07-

In [22]:
# Step 5: Evaluate using the stripped model
# - Use np.argmax for predictions
# - Print classification_report and confusion_matrix

y_pred_pruned = np.argmax(stripped_model.predict(X_test_scaled, verbose=0), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

pruned_acc = np.mean(y_pred_pruned == y_true)
print(f"Pruned model test accuracy: {pruned_acc:.4f}")

print("\nClassification report:\n", classification_report(y_true, y_pred_pruned))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred_pruned))

Pruned model test accuracy: 0.9444

Classification report:
               precision    recall  f1-score   support

           0       0.95      1.00      0.97        18
           1       0.91      0.95      0.93        21
           2       1.00      0.87      0.93        15

    accuracy                           0.94        54
   macro avg       0.95      0.94      0.94        54
weighted avg       0.95      0.94      0.94        54

Confusion matrix:
 [[18  0  0]
 [ 1 20  0]
 [ 0  2 13]]


## Problem 1 - Part (d)

### Knowledge Distillation

In [23]:
# Step 1: Define a Sequential model for Student with:
# - Dense(32, relu)
# - Dense(16, relu)
# - Dense(3, softmax)

student_model = Sequential([
    Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dense(16, activation='relu'),
    Dense(3, activation='softmax')
])

student_model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_9 (Dense)             (None, 32)                448       
                                                                 
 dense_10 (Dense)            (None, 16)                528       
                                                                 
 dense_11 (Dense)            (None, 3)                 51        
                                                                 
Total params: 1027 (4.01 KB)
Trainable params: 1027 (4.01 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [24]:
# Step 2: Use model.predict() on X_train_scaled to obtain teacher soft labels

teacher_preds_soft = model.predict(X_train_scaled, verbose=0)

print("Teacher soft labels shape:", teacher_preds_soft.shape)
print("Example soft label:", teacher_preds_soft[0])

Teacher soft labels shape: (124, 3)
Example soft label: [9.9818629e-01 1.6678566e-03 1.4593289e-04]


In [25]:
# Step 3:
# (a) Concatenate hard (y_train_cat) and soft (teacher_preds_soft) labels along axis=1
#     to create a combined label for distillation
y_train_combined = np.concatenate([y_train_cat, teacher_preds_soft], axis=1)
print("Combined label shape:", y_train_combined.shape)
# (b) Define a custom distillation_loss() function that:
#     - Splits y_true_combined into y_true_hard and y_true_soft
#     - Computes two losses (both using categorical_crossentropy)
#     - Combines them with a weight factor alpha = 0.5

# Hint: Use slicing [:, :3] and [:, 3:] to split the combined labels

alpha = 0.5

def distillation_loss(y_true_combined, y_pred):
    y_true_hard = y_true_combined[:, :3]
    y_true_soft = y_true_combined[:, 3:]

    # Hard-label loss: student predictions vs ground-truth labels
    hard_loss = tf.keras.losses.categorical_crossentropy(y_true_hard, y_pred)

    # Soft-label loss: student predictions vs teacher soft labels
    soft_loss = tf.keras.losses.categorical_crossentropy(y_true_soft, y_pred)

    # Weighted combination
    return alpha * hard_loss + (1.0 - alpha) * soft_loss

Combined label shape: (124, 6)


In [26]:
# Step 4: Compile the student model with Adam optimizer and distillation_loss
# - Train for 10 epochs, batch_size=8, validation_split=0.2

student_model.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

history_student = student_model.fit(
    X_train_scaled, y_train_combined,
    epochs=10,
    batch_size=8,
    validation_split=0.2,
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 0s 6ms/step - loss: 1.0471 - accuracy: 0.5152 - val_loss: 0.9839 - val_accuracy: 0.4400
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 0.8524 - accuracy: 0.6667 - val_loss: 0.8012 - val_accuracy: 0.6800
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.6951 - accuracy: 0.8081 - val_loss: 0.6592 - val_accuracy: 0.8000
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.5649 - accuracy: 0.8788 - val_loss: 0.5312 - val_accuracy: 0.9200
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.4554 - accuracy: 0.9394 - val_loss: 0.4331 - val_accuracy: 0.9600
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.3690 - accuracy: 0.9697 - val_loss: 0.3511 - val_accuracy: 0.9600
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.2966 - accuracy: 0.9697 - val_loss: 0.2852 - val_accuracy: 0.9600
Epoch 8/10
13/13 [==

In [27]:
# Step 5: Convert the student model to TFLite.
# - Save it as "model_kd.tflite".
# - Print the file size in KB.
converter = tf.lite.TFLiteConverter.from_keras_model(student_model)
tflite_kd = converter.convert()

with open("model_kd.tflite", "wb") as f:
    f.write(tflite_kd)

print(f"KD student TFLite model size: {file_size_kb('model_kd.tflite'):.2f} KB")


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpl_m5orwk/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpl_m5orwk/assets


KD student TFLite model size: 6.12 KB


2026-07-24 18:49:03.296195: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:49:03.296214: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:49:03.296308: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpl_m5orwk
2026-07-24 18:49:03.296745: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:49:03.296749: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpl_m5orwk
2026-07-24 18:49:03.297875: I tensorflow/cc/saved_model/loader.cc:233] Restoring SavedModel bundle.
2026-07-24 18:49:03.313894: I tensorflow/cc/saved_model/loader.cc:217] Running initialization op on SavedModel bundle at path: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpl_m5orwk
2026-07-

In [28]:
# Step 6: Use student_model.predict() to obtain predictions on X_test_scaled
# - Print classification_report and confusion_matrix

y_pred_kd = np.argmax(student_model.predict(X_test_scaled, verbose=0), axis=1)
y_true = np.argmax(y_test_cat, axis=1)

kd_acc = np.mean(y_pred_kd == y_true)
print(f"KD student test accuracy: {kd_acc:.4f}")

print("\nClassification report:\n", classification_report(y_true, y_pred_kd))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred_kd))


KD student test accuracy: 1.0000

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        18
           1       1.00      1.00      1.00        21
           2       1.00      1.00      1.00        15

    accuracy                           1.00        54
   macro avg       1.00      1.00      1.00        54
weighted avg       1.00      1.00      1.00        54

Confusion matrix:
 [[18  0  0]
 [ 0 21  0]
 [ 0  0 15]]


## Problem 1 - Part (e)

### Possibility of Further Model Size Reduction

Can you **further reduce the model size** beyond the smallest model obtained in parts **(b)**, **(c)**, or **(d)**, **without sacrificing significant classification performance**?

Your task is to:

1. **Analyze and compare** the results from previous parts: Which model had the smallest size? Which performed best?

2. **Propose a strategy** that combines or enhances techniques learned so far.

3. **Implement** your proposed solution.

4. **Evaluate** the resulting model using both:
   - TFLite model size (in KB)
   - Classification performance (accuracy and report)

5. **Justify your results:**
   - If further size reduction is **not** possible without major loss of accuracy, explain why.
   - If you succeed in reducing the size **further**, highlight what change made the biggest difference.


### **Note:** If this part includes any code, please include it below. The related discussion should be submitted as part of your PDF that contains answers to all [Dis] questions in this assignment.


In [30]:
#Prune the distilled student, fine-tuning with the same distillation loss
end_step_e = int(np.ceil(len(X_train_scaled) / batch_size)) * epochs_prune

pruning_schedule_e = tfmot.sparsity.keras.PolynomialDecay(
    initial_sparsity=0.5,
    final_sparsity=0.7,
    begin_step=0,
    end_step=end_step_e
)

# Wrap each trained student layer so pruning starts from the distilled weights
pruned_student = Sequential([
    prune_low_magnitude(
        Dense(32, activation='relu', input_shape=(X_train_scaled.shape[1],)),
        pruning_schedule=pruning_schedule_e
    ),
    prune_low_magnitude(Dense(16, activation='relu'), pruning_schedule=pruning_schedule_e),
    prune_low_magnitude(Dense(3, activation='softmax'), pruning_schedule=pruning_schedule_e)
])

# Build, then copy the trained student weights into the prunable model
pruned_student.build((None, X_train_scaled.shape[1]))
for pruned_layer, trained_layer in zip(pruned_student.layers, student_model.layers):
    pruned_layer.layer.set_weights(trained_layer.get_weights())

pruned_student.compile(
    optimizer='adam',
    loss=distillation_loss,
    metrics=['accuracy']
)

history_e = pruned_student.fit(
    X_train_scaled, y_train_combined,
    epochs=epochs_prune,
    batch_size=batch_size,
    validation_split=0.2,
    callbacks=[tfmot.sparsity.keras.UpdatePruningStep()],
    verbose=1
)

Epoch 1/10
13/13 [==============================] - 0s 6ms/step - loss: 0.1385 - accuracy: 0.9899 - val_loss: 0.1434 - val_accuracy: 1.0000
Epoch 2/10
13/13 [==============================] - 0s 1ms/step - loss: 0.1083 - accuracy: 0.9899 - val_loss: 0.1186 - val_accuracy: 1.0000
Epoch 3/10
13/13 [==============================] - 0s 1ms/step - loss: 0.0891 - accuracy: 0.9899 - val_loss: 0.1079 - val_accuracy: 1.0000
Epoch 4/10
13/13 [==============================] - 0s 1ms/step - loss: 0.0759 - accuracy: 0.9899 - val_loss: 0.0996 - val_accuracy: 1.0000
Epoch 5/10
13/13 [==============================] - 0s 1ms/step - loss: 0.0664 - accuracy: 0.9899 - val_loss: 0.0934 - val_accuracy: 1.0000
Epoch 6/10
13/13 [==============================] - 0s 1ms/step - loss: 0.0588 - accuracy: 0.9899 - val_loss: 0.0887 - val_accuracy: 1.0000
Epoch 7/10
13/13 [==============================] - 0s 1ms/step - loss: 0.0534 - accuracy: 0.9899 - val_loss: 0.0840 - val_accuracy: 1.0000
Epoch 8/10
13/13 [==

In [33]:
#Strip pruning wrappers, then apply full INT8 quantization
stripped_student = tfmot.sparsity.keras.strip_pruning(pruned_student)

converter = tf.lite.TFLiteConverter.from_keras_model(stripped_student)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = lambda: representative_data_gen(X_train_scaled)
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_final = converter.convert()
with open("model_final.tflite", "wb") as f:
    f.write(tflite_final)

print(f"Final (KD + pruning + INT8) model size: {file_size_kb('model_final.tflite'):.2f} KB")

INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpx0ml802a/assets


INFO:tensorflow:Assets written to: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpx0ml802a/assets


Final (KD + pruning + INT8) model size: 3.71 KB


/Users/kourosh/ai/projects/tinyml-arduino/lib/python3.11/site-packages/tensorflow/lite/python/convert.py:947: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
2026-07-24 18:58:23.506333: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:378] Ignored output_format.
2026-07-24 18:58:23.506342: W tensorflow/compiler/mlir/lite/python/tf_tfl_flatbuffer_helpers.cc:381] Ignored drop_control_dependency.
2026-07-24 18:58:23.506447: I tensorflow/cc/saved_model/reader.cc:83] Reading SavedModel from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpx0ml802a
2026-07-24 18:58:23.506739: I tensorflow/cc/saved_model/reader.cc:51] Reading meta graph with tags { serve }
2026-07-24 18:58:23.506743: I tensorflow/cc/saved_model/reader.cc:146] Reading SavedModel debug info (if present) from: /var/folders/wn/sc8rx18x2mv77cwtmrpd7jbc0000gn/T/tmpx0ml802a
2026-07-24 18:58:23.507405: I tensorflow/cc/saved_model/loader.cc:233] 

In [34]:
#Evaluate the final model
interpreter = tf.lite.Interpreter(model_path="model_final.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

y_pred_final = []
for i in range(len(X_test_scaled)):
    sample = X_test_scaled[i:i + 1].astype(np.float32)
    input_scale, input_zero_point = input_details["quantization"]
    sample = np.round(sample / input_scale + input_zero_point).astype(np.int8)

    interpreter.set_tensor(input_details["index"], sample)
    interpreter.invoke()
    output = interpreter.get_tensor(output_details["index"])[0]
    y_pred_final.append(np.argmax(output))

y_pred_final = np.array(y_pred_final)
y_true = np.argmax(y_test_cat, axis=1)

final_acc = np.mean(y_pred_final == y_true)
print(f"Final model test accuracy: {final_acc:.4f}")
print("\nClassification report:\n", classification_report(y_true, y_pred_final))
print("Confusion matrix:\n", confusion_matrix(y_true, y_pred_final))

Final model test accuracy: 0.9444

Classification report:
               precision    recall  f1-score   support

           0       1.00      0.94      0.97        18
           1       0.95      0.90      0.93        21
           2       0.88      1.00      0.94        15

    accuracy                           0.94        54
   macro avg       0.94      0.95      0.95        54
weighted avg       0.95      0.94      0.94        54

Confusion matrix:
 [[17  1  0]
 [ 0 19  2]
 [ 0  0 15]]


# Problem 2: Exploring Edge Impulse (20 points)


### Note

Problem 2 consists entirely of discussion questions. Submit your responses in the same PDF file that contains answers to the other **[Dis]** questions in this assignment.

Before submission, make sure this notebook runs with the **Python (tinyml-arduino)** kernel and that all requested outputs are visible. Host this notebook and your discussion PDF in your public GitHub repository, then submit the repository link through Canvas.
